# 00. Prepare Inputs

원본 강의 txt와 메타데이터 CSV를 읽어 5분 세그먼트 JSON, 전체 transcript JSON, TTS 입력 텍스트를 만든다.

In [ ]:
from google.colab import drive
from pathlib import Path
import csv, json, re

drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/tribe-v2-student-reaction')
TRANSCRIPT_DIR = ROOT / 'inputs' / 'transcripts'
METADATA_CSV = ROOT / 'inputs' / 'metadata' / '강의 메타데이터.csv'
OUTPUT_DIR = ROOT / 'outputs' / 'prepared'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PILOT_DATES = ['2026-02-02', '2026-02-09', '2026-02-24']
LINE_PATTERN = re.compile(r'<(\d{2}:\d{2}:\d{2})>\s+(\S+):\s+(.*)')
SEGMENT_SECONDS = 60 * 5

In [ ]:
def ts_to_seconds(ts: str) -> int:
    h, m, s = ts.split(':')
    return int(h) * 3600 + int(m) * 60 + int(s)

def parse_transcript(path: Path):
    lines = []
    offset = 0
    prev_raw = None
    for raw in path.read_text(encoding='utf-8').splitlines():
        m = LINE_PATTERN.match(raw.strip())
        if not m:
            continue
        timestamp, speaker, text = m.groups()
        raw_seconds = ts_to_seconds(timestamp)
        if prev_raw is not None and raw_seconds < prev_raw - 1800:
            offset += 12 * 3600
        prev_raw = raw_seconds
        lines.append({
            'timestamp': timestamp,
            'speaker': speaker,
            'text': text,
            'seconds': raw_seconds + offset,
        })
    return lines

def load_metadata():
    metadata = {}
    with open(METADATA_CSV, encoding='utf-8-sig') as f:
        for row in csv.DictReader(f):
            metadata.setdefault(row['date'], row)
    return metadata

In [ ]:
metadata = load_metadata()

for date in PILOT_DATES:
    transcript_path = TRANSCRIPT_DIR / f'{date}_kdt-backendj-21th.txt'
    lines = parse_transcript(transcript_path)
    start = lines[0]['seconds']
    end = lines[-1]['seconds']
    cursor = start
    segments = []
    while cursor <= end:
        window = [line for line in lines if cursor <= line['seconds'] < cursor + SEGMENT_SECONDS]
        if window:
            seg_id = f"seg-{len(segments) + 1:02d}"
            segments.append({
                'segment_id': seg_id,
                'start_time': window[0]['timestamp'],
                'end_time': window[-1]['timestamp'],
                'tts_text': ' '.join(line['text'] for line in window),
                'lines': [{k: v for k, v in line.items() if k != 'seconds'} for line in window],
            })
        cursor += SEGMENT_SECONDS

    save_dir = OUTPUT_DIR / date
    save_dir.mkdir(parents=True, exist_ok=True)
    (save_dir / 'segments.json').write_text(json.dumps(segments, ensure_ascii=False, indent=2), encoding='utf-8')
    (save_dir / 'transcript.json').write_text(json.dumps({'lecture_date': date, 'segments': segments}, ensure_ascii=False, indent=2), encoding='utf-8')
    (save_dir / 'metadata.json').write_text(json.dumps(metadata[date], ensure_ascii=False, indent=2), encoding='utf-8')
    print('prepared', date, len(segments))